In [ ]:
import pandas as pd
from pymilvus import Collection, CollectionSchema, FieldSchema, DataType, connections, utility
from FlagEmbedding import BGEM3FlagModel
from sentence_transformers import SentenceTransformer
import threading
import json
from jsonschema import validate, ValidationError

# Change the path of json schema. Another adjustment needed in load_json_schema()
JSON_SCHEMA = r"C:/Users/admin/Desktop/job_postings.schema.json"

with open(JSON_SCHEMA, encoding="utf-8") as f:
    JOB_POSTINGS_SCHEMA = json.load(f)

MODEL = {
    "bge-m3": {"loader": lambda: BGEM3FlagModel('BAAI/bge-m3', use_fp16=True), "dim": 1024},
    "all-MiniLM-L6-v2": {"loader": lambda: SentenceTransformer('all-MiniLM-L6-v2'), "dim": 384}
}


_model_cache = {}                   # lazy loading
_model_lock = threading.Lock()


def load_model(model_name="bge-m3"):
    with _model_lock:
        
        if model_name not in _model_cache:
            if model_name in MODEL:
                print(f"Loading embedding model: {model_name}")
                _model_cache[model_name] = MODEL[model_name]["loader"]()

            else:
                raise ValueError(f"Unsupported model: {model_name}")
            
        return _model_cache[model_name]



def generate_embeddings(texts, model_name="bge-m3"):

    if isinstance(texts, str):
        texts = [texts]

    model = load_model(model_name)


    if model_name == "bge-m3":
        embeddings = model.encode(
            texts,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False
        )["dense_vecs"].tolist()
    else:
        embeddings = model.encode(texts)


    # Flatten embeddings if it's a single vector wrapped in a list
    if len(embeddings) == 1:
        return embeddings[0]
    return embeddings



def connect_milvus(
    host="127.0.0.1",
    port="19530",
    user=None,
    password=None,
    secure=False,
    alias="default"
):

    if connections.has_connection(alias):
        print(f"Milvus connection '{alias}' already exists. Skipping reconnect.")
        
        return
    
    connect_params = {
        "host": host,
        "port": port,
        "secure": secure
    }


    if user and password:
        connect_params.update({"user": user, "password": password})
    
    connections.connect(alias, **connect_params)
    print(f"Connected to Milvus [{host}:{port}] {'(SSL enabled)' if secure else ''}")


def create_or_load_resume_collection(
    collection_name="resume",
    dim=1024,               # 1024 for 'BAAI/bge-m3'
    overwrite=False
):


    if overwrite and utility.has_collection(collection_name):
        print(f"Dropping existing collection: {collection_name}")
        utility.drop_collection(collection_name)

    if utility.has_collection(collection_name):
        print(f"Loading existing JD collection: {collection_name}")
        collection = Collection(name=collection_name)

    else:
        print(f"Creating new resume collection: {collection_name}")
        fields = [
            FieldSchema(name="resume_id", dtype=DataType.INT64, is_primary=True, auto_id=True),    # Unique resume ID
            FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=dim),                # Embedding vector
            FieldSchema(name="job_company", dtype=DataType.VARCHAR, max_length=200),            # Company name
            FieldSchema(name="job_title", dtype=DataType.VARCHAR, max_length=200),              # Job title
            FieldSchema(name="job_application_url", dtype=DataType.VARCHAR, max_length=500)     # Application URL
        ]

        schema = CollectionSchema(fields, description="Job postings with embeddings")
        collection = Collection(name=collection_name, schema=schema)
        collection.flush()
    # Build an index if nots already built over "embedding"
    if len(collection.indexes) == 0:
        print("Creating index on 'embedding' field...")
        index_params = {
            "index_type": "IVF_FLAT",
            "metric_type": "L2",
            "params": {"nlist": 128}
        }
        collection.create_index(
            field_name="embedding",
            index_params=index_params,
            index_name="embedding_index"
        )
        print("Index created.")
    else:
        print("Index already exists; skipping create_index.")


    return collection


def create_or_load_jd_collection(
    collection_name="job_postings",
    dim=1024,               # 1024 for 'BAAI/bge-m3'
    overwrite=False
):


    if overwrite and utility.has_collection(collection_name):
        print(f"Dropping existing collection: {collection_name}")
        utility.drop_collection(collection_name)

    if utility.has_collection(collection_name):
        print(f"Loading existing JD collection: {collection_name}")
        collection = Collection(name=collection_name)

    else:
        print(f"Creating new JD collection: {collection_name}")
        fields = [
            FieldSchema(name="job_id", dtype=DataType.INT64, is_primary=True, auto_id=True),    # Unique job ID
            FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=dim),                # Embedding vector
            FieldSchema(name="job_company", dtype=DataType.VARCHAR, max_length=200),            # Company name
            FieldSchema(name="job_title", dtype=DataType.VARCHAR, max_length=200),              # Job title
            FieldSchema(name="job_application_url", dtype=DataType.VARCHAR, max_length=500)     # Application URL
        ]

        schema = CollectionSchema(fields, description="Job postings with embeddings")
        collection = Collection(name=collection_name, schema=schema)
        collection.flush()
    # Build an index if nots already built over "embedding"
    if len(collection.indexes) == 0:
        print("Creating index on 'embedding' field...")
        index_params = {
            "index_type": "IVF_FLAT",
            "metric_type": "L2",
            "params": {"nlist": 128}
        }
        collection.create_index(
            field_name="embedding",
            index_params=index_params,
            index_name="embedding_index"
        )
        print("Index created.")
    else:
        print("Index already exists; skipping create_index.")


    return collection


# Change the path of json schema.
def load_json_schema(schema_path = r"C:/Users/windows/Downloads/seeu/team2/mine/job_postings.schema.json"):

    try:
        with open(schema_path, encoding="utf-8") as f:
            schema = json.load(f)
        print(f"Loaded schema from {schema_path}")
        return schema
    
    except FileNotFoundError:
        raise FileNotFoundError(f"{schema_path} file not found. Check the path.")
    
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON format in {schema_path}: {e}")


def insert_embeddings(
    job_postings_json,                      # JSON string
    collection_name="job_postings",
    model_name="bge-m3"
):

    try:
        job_postings = json.loads(job_postings_json)

    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON input: {e}")


    job_schema = load_json_schema()
    try:
        validate(instance=job_postings, schema=job_schema)
        print("JSON data matches the job postings schema.")

    except ValidationError as e:
        raise ValueError(f"JSON validation failed: {e.message}")


    connect_milvus()
    dim = MODEL[model_name]["dim"]
    collection = create_or_load_jd_collection(collection_name, dim=dim)

    if not job_postings:
        raise ValueError("job_postings list is empty.")

    # Generate embeddings;  LLM needed
    texts = [
        f"{job['job_company']} {job['job_title']} {job['job_description']}"
        for job in job_postings
    ]
    embeddings = generate_embeddings(texts, model_name=model_name)


    job_companies = [job["job_company"] for job in job_postings]
    job_titles = [job["job_title"] for job in job_postings]
    job_urls = [job["job_application_url"] for job in job_postings]

    # Insert into Milvus
    collection.insert([
        embeddings,
        job_companies,
        job_titles,
        job_urls
    ])
    collection.flush()

    print(f"Inserted {len(job_postings)} job_postings into collection '{collection_name}'.")



 
def search_embeddings(
    query_text, 
    collection_name="job_postings", 
    top_k=5, 
    model_name="bge-m3"
):

    connect_milvus()

    if not utility.has_collection(collection_name):
        raise RuntimeError(f"Collection '{collection_name}' does not exist.")

    collection = Collection(name=collection_name)
    collection.load()

    query_embedding = generate_embeddings(query_text, model_name=model_name)

 
    results = collection.search(
        data=[query_embedding],
        anns_field="embedding",
        param={"metric_type": "L2", "params": {"nprobe": 10}},
        limit=top_k,
        output_fields=["job_id", "job_company", "job_title", "job_application_url"]
    )


    matches = []
    for hit in results[0]:
        matches.append({
            "job_id": hit.entity.get("job_id"),
            "job_company": hit.entity.get("job_company"),
            "job_title": hit.entity.get("job_title"),
            "job_application_url": hit.entity.get("job_application_url"),
            "distance": hit.distance 
        })

    print(f"Found {len(matches)} matching job postings in '{collection_name}'.")

    return matches

# Example: Insert job postings into Milvus
json_file_path = "C:/Users/admin/Desktop/SeeU-Project-main/jobs.json"
with open(json_file_path, "r", encoding="utf-8") as f:
    job_postings_json = f.read()

insert_embeddings(job_postings_json, collection_name="job_postings", model_name="bge-m3")

# Example: Assume a summary has been generated from the resume information

query_text = "Experienced software engineer with Python and ML background"
matches = search_embeddings(query_text, collection_name="job_postings", top_k=5, model_name="bge-m3")
print(matches)

# Save matches to CSV
matches_df = pd.DataFrame(matches)
matches_df.to_csv("milvus_matching_results.csv", index=False)
print("Matching results saved to milvus_matching_results.csv")


ImportError: dlopen(/opt/anaconda3/lib/python3.8/site-packages/scipy/sparse/linalg/_isolve/_iterative.cpython-38-darwin.so, 0x0002): Library not loaded: @rpath/liblapack.3.dylib
  Referenced from: <BEFB6E07-597A-3758-A60B-141E8E215EF0> /opt/anaconda3/lib/python3.8/site-packages/scipy/sparse/linalg/_isolve/_iterative.cpython-38-darwin.so
  Reason: tried: '/opt/anaconda3/lib/python3.8/site-packages/scipy/sparse/linalg/_isolve/liblapack.3.dylib' (no such file), '/opt/anaconda3/lib/python3.8/site-packages/scipy/sparse/linalg/_isolve/../../../../../../liblapack.3.dylib' (no such file), '/opt/anaconda3/lib/python3.8/site-packages/scipy/sparse/linalg/_isolve/liblapack.3.dylib' (no such file), '/opt/anaconda3/lib/python3.8/site-packages/scipy/sparse/linalg/_isolve/../../../../../../liblapack.3.dylib' (no such file), '/opt/anaconda3/bin/../lib/liblapack.3.dylib' (no such file), '/opt/anaconda3/bin/../lib/liblapack.3.dylib' (no such file), '/usr/local/lib/liblapack.3.dylib' (no such file), '/usr